In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nmslib
import pickle
import time
import os
import psutil
from tqdm import tqdm

device  = torch.device('cuda:2')
THREADS = 32

class NeuralMinHashEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, n_hashes=512):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Linear(in_dim,     hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Identity(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        self.hash_proj = nn.Linear(hidden_dim, n_hashes)
        self.bn_out    = nn.BatchNorm1d(n_hashes)

    def forward(self, x):
        x   = torch.log1p(x * 1e6)
        out = self.extractor(x)
        out = self.hash_proj(out)
        out = self.bn_out(out)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

def recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

# ─── Load data ────────────────────────────────────────────────────────────────
qt_full = np.load('/tmp/qtree_vectors_full.npy')
with open('/tmp/gt_lookup_full.pkl', 'rb') as f: gt_full = pickle.load(f)

QUERY_START_ID = 187019
corpus_qt_full = qt_full[:QUERY_START_ID]
query_qt_full  = qt_full[QUERY_START_ID:]

# ─── Load model ───────────────────────────────────────────────────────────────
model = NeuralMinHashEncoder(in_dim=qt_full.shape[1]).to(device)
model.load_state_dict(torch.load('/tmp/best_minhash_full.pt',
                                  weights_only=True, map_location=device))
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} params")

# ─── Embedding quality check ──────────────────────────────────────────────────
gt_sims, rand_sims = [], []
for i in range(200):
    qid    = QUERY_START_ID + i
    pos_id = gt_full.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_ID)
    q = torch.tensor(qt_full[qid],     dtype=torch.float32).unsqueeze(0).to(device)
    p = torch.tensor(qt_full[pos_id],  dtype=torch.float32).unsqueeze(0).to(device)
    r = torch.tensor(qt_full[rand_id], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        eq, ep, er = model(q), model(p), model(r)
    wj_gt   = float(torch.minimum(eq,ep).sum() / torch.maximum(eq,ep).sum())
    wj_rand = float(torch.minimum(eq,er).sum() / torch.maximum(eq,er).sum())
    gt_sims.append(wj_gt); rand_sims.append(wj_rand)
print(f"GT WJ sim:   {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Rand WJ sim: {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"Gap:         {np.mean(gt_sims)-np.mean(rand_sims):.4f}")

# ─── Generate embeddings ──────────────────────────────────────────────────────
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_full), 512), desc="Embedding"):
        batch = torch.tensor(qt_full[start:start+512],
                             dtype=torch.float32).to(device)
        all_embs.append(model(batch).cpu().numpy())

embs_full   = np.vstack(all_embs)
corpus_full = embs_full[:QUERY_START_ID]
query_full  = embs_full[QUERY_START_ID:]
vec_mb      = corpus_full.nbytes / 1024**2
print(f"Embeddings: {embs_full.shape} | Non-neg: {(embs_full>=0).all()} | "
      f"Vec: {vec_mb:.1f} MB")

# ─── Build WJ index ───────────────────────────────────────────────────────────
print("Building WJ index...")
m0  = get_mem_mb()
idx = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in tqdm(range(len(corpus_full)), desc="Adding", mininterval=2.0):
    idx.addDataPoint(i, corpus_full[i])
t0 = time.time()
idx.createIndex({'M':20,'efConstruction':200,'post':1}, print_progress=True)
build_s = time.time() - t0
idx_mb  = get_mem_mb() - m0
idx.setQueryTimeParams({'efSearch': 200})
print(f"Build: {build_s:.1f}s | Idx mem: {idx_mb:.1f} MB")

# ─── Query K=500 ──────────────────────────────────────────────────────────────
t0   = time.time()
nbrs = idx.knnQueryBatch(query_full, k=500, num_threads=THREADS)
qps  = len(query_full) / (time.time() - t0)
rec  = {k: recall_at_k(gt_full, nbrs, QUERY_START_ID, k)
        for k in [10,50,100,500]}

print(f"\nQPS: {qps:.1f}")
for k,r in rec.items(): print(f"  R@{k} = {r:.4f}")

res_full = {**rec, 'qps': qps, 'build_s': build_s,
            'vec_mb': vec_mb, 'idx_mb': idx_mb}

# ─── Update saved results ─────────────────────────────────────────────────────
with open('/tmp/results_minhash.pkl', 'rb') as f:
    saved = pickle.load(f)
saved['full'] = res_full
with open('/tmp/results_minhash.pkl', 'wb') as f:
    pickle.dump(saved, f)
print(f"\nResults updated in /tmp/results_minhash.pkl")

Model loaded: 20,237,824 params
GT WJ sim:   0.6864 ± 0.1285
Rand WJ sim: 0.0352 ± 0.0393
Gap:         0.6512


Embedding: 100%|██████████| 457/457 [00:07<00:00, 65.05it/s]


Embeddings: (233773, 512) | Non-neg: True | Vec: 365.3 MB
Building WJ index...


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 566682.61it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Build: 36.6s | Idx mem: 631.8 MB

QPS: 4503.9
  R@10 = 0.5821
  R@50 = 0.6604
  R@100 = 0.6729
  R@500 = 0.7068

Results updated in /tmp/results_minhash.pkl


In [2]:
def wj_triplet_loss(anchors, positives, margin=0.3):
    """
    Triplet loss in WJ space.
    anchors, positives: already ReLU + L1 normalized (WJ compatible)
    """
    # WJ similarity: Σmin(a,b) / Σmax(a,b)
    sim_ap = (torch.minimum(anchors, positives).sum(dim=1) /
              torch.maximum(anchors, positives).sum(dim=1).clamp(min=1e-10))

    # In-batch hard negatives: most similar positive from different query
    # Cross similarity matrix (B, B)
    mins_cross = torch.min(
        anchors.unsqueeze(1),    # (B, 1, D)
        positives.unsqueeze(0)   # (1, B, D)
    ).sum(dim=2)                 # (B, B)
    maxs_cross = torch.max(
        anchors.unsqueeze(1),
        positives.unsqueeze(0)
    ).sum(dim=2)                 # (B, B)
    sim_cross = mins_cross / maxs_cross.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    sim_an = sim_cross.max(dim=1).values  # hardest negative

    # Want sim_ap > sim_an + margin
    loss    = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[violated].mean(), violated.sum().item()

# Retrain with correct loss
model_mh_full = NeuralMinHashEncoder(in_dim=IN_DIM).to(device)
if torch.cuda.device_count() > 1:
    model_par = nn.DataParallel(model_mh_full, device_ids=[2,3,4,5,6,7])
else:
    model_par = model_mh_full

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_mh_full.parameters(),
                               lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_par.train()
    total_loss = 0.0; total_steps = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_par(combined)  # already WJ normalized
        a_emb     = out[:B]
        p_emb     = out[B:]

        loss, n_violated = wj_triplet_loss(a_emb, p_emb, margin=0.3)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mh_full.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item(); total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'violated': n_violated})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_mh_full.state_dict(),
                   '/tmp/best_minhash_full.pt')

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

NameError: name 'IN_DIM' is not defined

In [5]:
# Neural MinHash + exact original-WJ rerank
# This turns Neural MinHash into the same two-stage pattern used by MLP+cosine:
#   Neural MinHash WJ index -> top-K candidates -> exact WJ rerank on original quadtree vectors.

import gc

mh_rerank_dataset = "full"      # "10k", "full", or "both"
mh_candidate_ks = [100, 200, 500, 1000]
mh_rerank_batch_size = 16      # use 8 or 4 for full if GPU memory is tight
mh_out_path = "/tmp/results_minhash_rerank.pkl"


def mh_load_dataset(name):
    if name == "10k":
        qt = np.load('/tmp/qt_10k.npy')
        with open('/tmp/gt_lookup_10k.pkl', 'rb') as f:
            gt = pickle.load(f)
        query_start = 8000
        ckpt = '/tmp/best_minhash_10k.pt'
        # If no dedicated 10k minhash checkpoint exists, this cell will tell us.
    elif name == "full":
        qt = np.load('/tmp/qtree_vectors_full.npy')
        with open('/tmp/gt_lookup_full.pkl', 'rb') as f:
            gt = pickle.load(f)
        query_start = 187019
        ckpt = '/tmp/best_minhash_full.pt'
    else:
        raise ValueError(name)
    return qt, gt, query_start, ckpt


def mh_recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt = set(gt_lookup.get(qid, [])[:K])
        if not gt:
            continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0


def mh_eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {k: mh_recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in [10, 50, 100, 500] if k <= max_k}


def mh_generate_embeddings(model, qt, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(qt), batch_size), desc="Embedding"):
            batch = torch.tensor(qt[start:start + batch_size], dtype=torch.float32).to(device)
            chunks.append(model(batch).cpu().numpy())
    return np.vstack(chunks)


def mh_build_wj_index(corpus_embs):
    m0 = get_mem_mb()
    idx = nmslib.init(method='hnsw', space='WeightedJaccard')
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({'efSearch': 200})
    return idx, build_s, idx_mb


def mh_rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU original-WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[absolute_i] for absolute_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])
    del corpus_t, corpus_sums_t
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return reranked


In [6]:
# Run Neural MinHash rerank experiment

def run_minhash_rerank(name):
    qt, gt, query_start, ckpt = mh_load_dataset(name)
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Missing checkpoint for {name}: {ckpt}")

    corpus_qt = qt[:query_start]
    query_qt = qt[query_start:]
    corpus_sums = corpus_qt.sum(axis=1)

    model = NeuralMinHashEncoder(in_dim=qt.shape[1]).to(device)
    model.load_state_dict(torch.load(ckpt, weights_only=True, map_location=device))
    model.eval()

    print("\n" + "=" * 80)
    print(f"Neural MinHash WJ candidates + exact original-WJ rerank -- {name}")
    print("=" * 80)
    print(f"corpus={corpus_qt.shape} | queries={query_qt.shape}")

    embs = mh_generate_embeddings(model, qt, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    vec_mb = corpus_embs.nbytes / 1024**2
    idx, build_s, idx_mb = mh_build_wj_index(corpus_embs)

    results = {}
    for k in mh_candidate_ks:
        t0 = time.time()
        nbrs_raw = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0
        rec_no = mh_eval_recall(gt, nbrs_raw, query_start, max_k=k)

        t0 = time.time()
        nbrs_rr = mh_rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums,
                                   device, batch_size=mh_rerank_batch_size)
        rerank_s = time.time() - t0
        rec_rr = mh_eval_recall(gt, nbrs_rr, query_start, max_k=k)
        qps = len(query_embs) / (hnsw_s + rerank_s)

        results[f'k{k}_orig_wj_rerank'] = {
            **rec_rr,
            'no_rerank': rec_no,
            'candidate_k': k,
            'qps': qps,
            'hnsw_s': hnsw_s,
            'rerank_s': rerank_s,
            'build_s': build_s,
            'vec_mb': vec_mb,
            'idx_mb': idx_mb,
        }
        print(f"K={k} | QPS={qps:.1f} | HNSW={hnsw_s:.2f}s | WJ={rerank_s:.2f}s")
        print("  no-rerank:", {kk: round(v, 4) for kk, v in rec_no.items()})
        print("  rerank:   ", {kk: round(v, 4) for kk, v in rec_rr.items()})

        del nbrs_raw, nbrs_rr
        gc.collect()

    return results

mh_all_results = {}
for ds_name in (["10k", "full"] if mh_rerank_dataset == "both" else [mh_rerank_dataset]):
    mh_all_results[ds_name] = run_minhash_rerank(ds_name)

try:
    with open(mh_out_path, 'rb') as f:
        saved = pickle.load(f)
except FileNotFoundError:
    saved = {'runs': {}}
run_key = time.strftime(f"{mh_rerank_dataset}_minhash_rerank_%Y%m%d_%H%M%S")
saved.setdefault('runs', {})[run_key] = {
    'config': {
        'dataset': mh_rerank_dataset,
        'candidate_ks': mh_candidate_ks,
        'rerank_batch_size': mh_rerank_batch_size,
    },
    'results': mh_all_results,
}
with open(mh_out_path, 'wb') as f:
    pickle.dump(saved, f)
print(f"\nSaved {run_key} to {mh_out_path}")



Neural MinHash WJ candidates + exact original-WJ rerank -- full
corpus=(187019, 18220) | queries=(46754, 18220)


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 706118.69it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
GPU original-WJ rerank: 100%|██████████| 2923/2923 [00:10<00:00, 278.59it/s]


K=100 | QPS=2009.8 | HNSW=11.70s | WJ=11.56s
  no-rerank: {10: 0.5822, 50: 0.6604, 100: 0.6729}
  rerank:    {10: 0.9701, 50: 0.85, 100: 0.6729}


GPU original-WJ rerank: 100%|██████████| 2923/2923 [00:08<00:00, 354.13it/s]


K=200 | QPS=1845.2 | HNSW=15.66s | WJ=9.68s
  no-rerank: {10: 0.5822, 50: 0.6604, 100: 0.6729}
  rerank:    {10: 0.9864, 50: 0.944, 100: 0.8606}


GPU original-WJ rerank: 100%|██████████| 2923/2923 [00:13<00:00, 210.28it/s]


K=500 | QPS=1345.3 | HNSW=19.76s | WJ=15.00s
  no-rerank: {10: 0.5822, 50: 0.6604, 100: 0.6729, 500: 0.7068}
  rerank:    {10: 0.9916, 50: 0.9861, 100: 0.9648, 500: 0.7068}


GPU original-WJ rerank: 100%|██████████| 2923/2923 [00:22<00:00, 132.80it/s]


K=1000 | QPS=956.9 | HNSW=25.74s | WJ=23.12s
  no-rerank: {10: 0.5822, 50: 0.6604, 100: 0.6729, 500: 0.7068}
  rerank:    {10: 0.9922, 50: 0.992, 100: 0.9841, 500: 0.8382}

Saved full_minhash_rerank_20260427_170041 to /tmp/results_minhash_rerank.pkl
